In [ ]:
# Install library Transformers dari Hugging Face untuk Pretrained ViT
!pip install transformers
!pip install keras-hub


In [ ]:
!unzip dataset.zip

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16, ResNet50

# Buat folder untuk menyimpan hasil
os.makedirs('results', exist_ok=True)

# Konfigurasi Dataset
IMG_HEIGHT, IMG_WIDTH = 128, 128
BATCH_SIZE = 32
NUM_CLASSES = 10 # Ganti jika jumlah kelas penyakit pada folder Anda berbeda
EPOCHS = 30

print(f"TensorFlow Version: {tf.__version__}")


TensorFlow Version: 2.20.0


In [ ]:
# --- 1. Custom VGG ---
def build_custom_vgg(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3), num_classes=NUM_CLASSES):
    inputs = layers.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = layers.Rescaling(1./255)(x)

    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(x)
    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(x)
    x = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs, outputs, name="Custom_VGG")

# --- 2. Custom ResNet ---
def residual_block(x, filters, stride=1):
    shortcut = x
    x = layers.Conv2D(filters, 3, strides=stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, strides=1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

def build_custom_resnet(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3), num_classes=NUM_CLASSES):
    inputs = layers.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = layers.Rescaling(1./255)(x)

    x = layers.Conv2D(32, 3, strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(pool_size=3, strides=2, padding='same')(x)

    x = residual_block(x, 32)
    x = residual_block(x, 64, stride=2)
    x = residual_block(x, 128, stride=2)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs, outputs, name="Custom_ResNet")

# --- 3. Custom ViT ---
def build_custom_vit(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3), num_classes=NUM_CLASSES, patch_size=16, embed_dim=64, num_heads=4, transformer_layers=4):
    inputs = layers.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = layers.Rescaling(1./255)(x)

    x = layers.Conv2D(embed_dim, kernel_size=patch_size, strides=patch_size, padding="VALID")(x)
    seq_len = (input_shape[0] // patch_size) * (input_shape[1] // patch_size)
    x = layers.Reshape((seq_len, embed_dim))(x)

    positions = tf.range(start=0, limit=seq_len, delta=1)
    pos_emb = layers.Embedding(input_dim=seq_len, output_dim=embed_dim)(positions)
    x = x + pos_emb

    for _ in range(transformer_layers):
        x1 = layers.LayerNormalization(epsilon=1e-6)(x)
        attention_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=0.1)(x1, x1)
        x2 = layers.Add()([attention_output, x])

        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        x3 = layers.Dense(embed_dim * 2, activation=tf.nn.gelu)(x3)
        x3 = layers.Dropout(0.1)(x3)
        x3 = layers.Dense(embed_dim)(x3)
        x3 = layers.Dropout(0.1)(x3)

        x = layers.Add()([x3, x2])

    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return models.Model(inputs, outputs, name="Custom_ViT")

# --- 4. Pretrained VGG & ResNet ---
def build_pretrained_model(model_type='vgg16', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3), num_classes=NUM_CLASSES):
    inputs = layers.Input(shape=input_shape)
    x = data_augmentation(inputs)

    if model_type == 'vgg16':
        base_model = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)
        x = tf.keras.applications.vgg16.preprocess_input(x)
    elif model_type == 'resnet50':
        base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
        x = tf.keras.applications.resnet50.preprocess_input(x)

    base_model.trainable = False
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs, outputs, name=f"Pretrained_{model_type.upper()}")

# --- 5. Pretrained ViT (KerasHub Native) ---
def build_pretrained_vit_hf(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3), num_classes=NUM_CLASSES):
    import keras_hub

    inputs = layers.Input(shape=input_shape)
    x = data_augmentation(inputs)

    # Model ViT bawaan KerasHub optimal di ukuran 224x224
    x = layers.Resizing(224, 224)(x)
    # Preprocessing standar ViT (nilai pixel antara 0 hingga 1)
    x = layers.Rescaling(1./255)(x)

    # Download Pretrained ViT Backbone (Sudah diperbaiki dengan link HF)
    vit_backbone = keras_hub.models.ViTBackbone.from_preset(
        "hf://google/vit-base-patch16-224",
        trainable=False
    )

    # Ekstraksi fitur
    x = vit_backbone(x)

    # Ratakan fitur (karena output berupa token sequences)
    x = layers.GlobalAveragePooling1D()(x)

    # Classification Head khusus untuk dataset
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs, outputs, name="Pretrained_ViT_KerasHub")

# --- 6. Helper Function Evaluasi & Training ---
def train_and_evaluate(model, train_data, val_data, epochs=EPOCHS):
    print(f"\n{'='*50}\nMemulai Training: {model.name}\n{'='*50}")
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    callbacks = [
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(filepath=f"results/{model.name}_best.keras", save_best_only=True)
    ]
    history = model.fit(train_data, validation_data=val_data, epochs=epochs, callbacks=callbacks)

    # Plot Accuracy & Loss
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Acc'); plt.plot(history.history['val_accuracy'], label='Val Acc')
    plt.title(f'Accuracy: {model.name}'); plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss'); plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title(f'Loss: {model.name}'); plt.legend()
    plt.savefig(f"results/{model.name}_history.png"); plt.show()

    # Classification Report & Confusion Matrix
    y_true, y_pred = [], []
    for images, labels in val_data:
        preds = model.predict(images, verbose=0)
        y_true.extend(np.argmax(labels.numpy(), axis=1))
        y_pred.extend(np.argmax(preds, axis=1))

    print(classification_report(y_true, y_pred, target_names=class_names))
    plt.figure(figsize=(10, 8))
    sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix: {model.name}')
    plt.savefig(f"results/{model.name}_confusion_matrix.png"); plt.show()


In [ ]:
# 1. Custom VGG
# model_vgg = build_custom_vgg()
# train_and_evaluate(model_vgg, train_dataset, val_dataset)

# 2. Custom ResNet
# model_resnet = build_custom_resnet()
# train_and_evaluate(model_resnet, train_dataset, val_dataset)

# 3. Custom ViT
# model_vit_custom = build_custom_vit()
# train_and_evaluate(model_vit_custom, train_dataset, val_dataset)

# 4. Pretrained VGG16
# model_pre_vgg = build_pretrained_model('vgg16')
# train_and_evaluate(model_pre_vgg, train_dataset, val_dataset)

# 5. Pretrained ResNet50
# model_pre_resnet = build_pretrained_model('resnet50')
# train_and_evaluate(model_pre_resnet, train_dataset, val_dataset)

# 6. Pretrained ViT (Versi Hugging Face) - Hidupkan dengan menghapus tanda pagar
model_pre_vit = build_pretrained_vit_hf()
train_and_evaluate(model_pre_vit, train_dataset, val_dataset)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]


Memulai Training: Pretrained_ViT_KerasHub
Epoch 1/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 565s 2s/step - accuracy: 0.0988 - loss: 2.3226 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 2/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 548s 2s/step - accuracy: 0.0932 - loss: 2.3028 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 3/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 556s 2s/step - accuracy: 0.0903 - loss: 2.3028 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 4/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 551s 2s/step - accuracy: 0.0903 - loss: 2.3028 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 5/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 554s 2s/step - accuracy: 0.0884 - loss: 2.3028 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 6/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 546s 2s/step - accuracy: 0.0881 - loss: 2.3028 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 7/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 484s 2s/step - accuracy: 0.0888 - loss: 2.3028 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 8/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 544s 2s/step -

KeyboardInterrupt: 